# NEXUS Downloaded Data Profiling Sandbox

Notebook này phân tích dữ liệu đã tải về trong `runtime/downloads/` trước khi đưa vào Raw/Bronze/Silver/Gold.

Mục tiêu:

1. Source Inventory & Volume
2. Heterogeneity
3. Schema & Data Type Profiling
4. Data Quality theo 6 dimensions: accuracy, completeness, consistency, timeliness, uniqueness, validity
5. Temporal Coverage, Freshness & Latency
6. Missing Data & Gap Analysis
7. Spatial Coverage & Joinability

Output được lưu vào `runtime/analysis/downloaded_data_profile/analysis_id=<timestamp>/` để tiện so sánh về sau.

In [1]:
from __future__ import annotations

import csv
import json
import math
import re
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

try:
    import pandas as pd
except ImportError:  # Notebook vẫn chạy được bằng stdlib.
    pd = None

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'runtime' / 'downloads').exists():
    PROJECT_ROOT = Path.cwd().parent

DOWNLOADS_DIR = PROJECT_ROOT / 'runtime' / 'downloads'
ANALYSIS_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUTPUT_DIR = PROJECT_ROOT / 'runtime' / 'analysis' / 'downloaded_data_profile' / f'analysis_id={ANALYSIS_ID}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Set RUN_ID_FILTER='20260520T034335Z' nếu muốn chỉ phân tích một run cụ thể.
RUN_ID_FILTER: str | None = None

# Sampling để notebook chạy nhanh dù raw data lớn.
MAX_FILES_PER_SOURCE = 60
MAX_RECORDS_PER_FILE = 1000
MAX_FIELD_EXAMPLES = 5
MAX_FLATTEN_DEPTH = 5

print(f'Project root: {PROJECT_ROOT}')
print(f'Downloads dir: {DOWNLOADS_DIR}')
print(f'Analysis output: {OUTPUT_DIR}')

Project root: d:\.Kỳ II năm Ba\Big Data\NEXUS
Downloads dir: d:\.Kỳ II năm Ba\Big Data\NEXUS\runtime\downloads
Analysis output: d:\.Kỳ II năm Ba\Big Data\NEXUS\runtime\analysis\downloaded_data_profile\analysis_id=20260521T045042Z


## Utility Functions

Các hàm dưới đây đọc profile/checkpoint/raw sample, infer schema, flatten nested fields, parse timestamp và lưu bảng ra CSV/JSON.

In [2]:
def read_json(path: Path, default: Any = None) -> Any:
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        return default


def read_jsonl(path: Path, limit: int | None = None) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    with path.open('r', encoding='utf-8', errors='replace') as file:
        for line in file:
            if not line.strip():
                continue
            try:
                value = json.loads(line)
                if isinstance(value, dict):
                    rows.append(value)
            except json.JSONDecodeError:
                continue
            if limit and len(rows) >= limit:
                break
    return rows


def read_csv_sample(path: Path, limit: int | None = None) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    with path.open('r', encoding='utf-8-sig', errors='replace', newline='') as file:
        reader = csv.DictReader(file)
        for row in reader:
            rows.append(dict(row))
            if limit and len(rows) >= limit:
                break
    return rows


def write_json(path: Path, payload: Any) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding='utf-8')
    return path


def write_csv(path: Path, rows: list[dict[str, Any]]) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    fieldnames: list[str] = []
    seen: set[str] = set()
    for row in rows:
        for key in row:
            if key not in seen:
                seen.add(key)
                fieldnames.append(key)
    with path.open('w', encoding='utf-8', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({key: serialize_cell(row.get(key)) for key in fieldnames})
    return path


def save_table(name: str, rows: list[dict[str, Any]]) -> tuple[Path, Path]:
    csv_path = write_csv(OUTPUT_DIR / f'{name}.csv', rows)
    json_path = write_json(OUTPUT_DIR / f'{name}.json', rows)
    return csv_path, json_path


def serialize_cell(value: Any) -> Any:
    if isinstance(value, (dict, list, tuple, set)):
        return json.dumps(value, ensure_ascii=False, sort_keys=True)
    return value


def show_table(rows: list[dict[str, Any]], n: int = 20) -> None:
    if pd is not None:
        display(pd.DataFrame(rows).head(n))
    else:
        print(json.dumps(rows[:n], indent=2, ensure_ascii=False))


def run_id_from_path(path: Path) -> str:
    for part in path.parts:
        if part.startswith('run_id='):
            return part.removeprefix('run_id=')
    return ''


def source_id_from_profile_path(path: Path) -> str:
    # runtime/downloads/<source_id>/run_id=.../metadata/profile.json
    try:
        return path.parents[2].name
    except IndexError:
        return ''


def latest_profile_paths(downloads_dir: Path, run_id_filter: str | None = None) -> list[Path]:
    profile_paths = sorted(downloads_dir.glob('*/run_id=*/metadata/profile.json'))
    if run_id_filter:
        return [p for p in profile_paths if run_id_from_path(p) == run_id_filter]
    by_source: dict[str, Path] = {}
    for path in profile_paths:
        source_id = source_id_from_profile_path(path)
        previous = by_source.get(source_id)
        if run_id_from_path(path) == 'consolidated':
            by_source[source_id] = path
            continue
        if previous is not None and run_id_from_path(previous) == 'consolidated':
            continue
        if previous is None or path.stat().st_mtime > previous.stat().st_mtime:
            by_source[source_id] = path
    return sorted(by_source.values())


def raw_dir_for_profile(profile_path: Path) -> Path:
    return profile_path.parents[1] / 'raw'


def metadata_dir_for_profile(profile_path: Path) -> Path:
    return profile_path.parents[1] / 'metadata'


def sample_raw_files(raw_dir: Path, max_files: int = MAX_FILES_PER_SOURCE) -> list[Path]:
    if not raw_dir.exists():
        return []
    files = [p for p in raw_dir.rglob('*') if p.is_file() and not p.name.endswith('.part')]
    priority = {'.jsonl': 0, '.json': 1, '.csv': 2}
    files = sorted(files, key=lambda p: (priority.get(p.suffix.lower(), 9), str(p)))
    return files[:max_files]


def infer_format(path: Path) -> str:
    suffix = path.suffix.lower()
    if suffix == '.jsonl':
        return 'jsonl'
    if suffix == '.json':
        return 'json'
    if suffix == '.csv':
        return 'csv'
    return suffix.removeprefix('.') or 'unknown'


def records_from_json_payload(payload: Any, source_id: str) -> list[dict[str, Any]]:
    if isinstance(payload, dict):
        hourly = payload.get('hourly')
        if isinstance(hourly, dict) and isinstance(hourly.get('time'), list):
            rows: list[dict[str, Any]] = []
            times = hourly.get('time', [])[:MAX_RECORDS_PER_FILE]
            for idx, observed_at in enumerate(times):
                row = {
                    'time': observed_at,
                    'latitude': payload.get('latitude'),
                    'longitude': payload.get('longitude'),
                    '_shape': 'openmeteo_hourly_arrays',
                }
                for key, value in hourly.items():
                    if key == 'time':
                        continue
                    if isinstance(value, list) and idx < len(value):
                        row[key] = value[idx]
                rows.append(row)
            return rows
        for key in ('results', 'data', 'records', 'items', 'features', 'rows', 'readings', 'feed'):
            value = payload.get(key)
            if isinstance(value, list):
                return [item for item in value[:MAX_RECORDS_PER_FILE] if isinstance(item, dict)]
            if isinstance(value, dict):
                nested = records_from_json_payload(value, source_id)
                if nested:
                    return nested
        longest = longest_list_of_dicts(payload)
        if longest:
            return longest[:MAX_RECORDS_PER_FILE]
        return [payload]
    if isinstance(payload, list):
        return [item for item in payload[:MAX_RECORDS_PER_FILE] if isinstance(item, dict)]
    return []


def longest_list_of_dicts(value: Any) -> list[dict[str, Any]]:
    candidates: list[list[dict[str, Any]]] = []
    def walk(item: Any) -> None:
        if isinstance(item, list) and item and all(isinstance(x, dict) for x in item[:10]):
            candidates.append(item)
        elif isinstance(item, dict):
            for child in item.values():
                walk(child)
        elif isinstance(item, list):
            for child in item[:10]:
                walk(child)
    walk(value)
    return max(candidates, key=len) if candidates else []


def read_records_from_file(path: Path, source_id: str) -> list[dict[str, Any]]:
    fmt = infer_format(path)
    if fmt == 'csv':
        return read_csv_sample(path, MAX_RECORDS_PER_FILE)
    if fmt == 'jsonl':
        return read_jsonl(path, MAX_RECORDS_PER_FILE)
    if fmt == 'json':
        payload = read_json(path)
        return records_from_json_payload(payload, source_id)
    return []


def flatten(value: Any, prefix: str = '', depth: int = 0, max_depth: int = MAX_FLATTEN_DEPTH) -> dict[str, Any]:
    if depth > max_depth:
        return {prefix or '<root>': '<max_depth>'}
    if isinstance(value, dict):
        out: dict[str, Any] = {}
        for key, child in value.items():
            child_key = f'{prefix}.{key}' if prefix else str(key)
            out.update(flatten(child, child_key, depth + 1, max_depth))
        return out or {prefix or '<root>': {}}
    if isinstance(value, list):
        out = {prefix or '<root>': value}
        if value and isinstance(value[0], dict):
            for idx, item in enumerate(value[:3]):
                out.update(flatten(item, f'{prefix}[]', depth + 1, max_depth))
        return out
    return {prefix or '<root>': value}


def type_name(value: Any) -> str:
    if value is None:
        return 'null'
    if isinstance(value, bool):
        return 'boolean'
    if isinstance(value, int) and not isinstance(value, bool):
        return 'integer'
    if isinstance(value, float):
        return 'number'
    if isinstance(value, list):
        if not value:
            return 'array<empty>'
        inner = sorted({type_name(item) for item in value[:20]})
        return f"array<{','.join(inner)}>"
    if isinstance(value, dict):
        return 'object'
    if isinstance(value, str):
        if value == '':
            return 'empty_string'
        if parse_datetime(value):
            return 'datetime_string'
        if looks_numeric(value):
            return 'numeric_string'
        return 'string'
    return type(value).__name__


def looks_numeric(value: str) -> bool:
    try:
        float(value)
        return True
    except Exception:
        return False


def parse_datetime(value: Any) -> datetime | None:
    if isinstance(value, datetime):
        return value if value.tzinfo else value.replace(tzinfo=timezone.utc)
    if not isinstance(value, str):
        return None
    text = value.strip()
    if not re.match(r'^\d{4}-\d{2}-\d{2}', text):
        return None
    normalized = text.replace('Z', '+00:00')
    for fmt_value in (normalized, normalized.replace(' ', 'T')):
        try:
            parsed = datetime.fromisoformat(fmt_value)
            return parsed if parsed.tzinfo else parsed.replace(tzinfo=timezone.utc)
        except Exception:
            pass
    for fmt in ('%Y-%m-%d %H:%M:%S', '%Y-%m-%d %H:%M', '%Y-%m-%d'):
        try:
            return datetime.strptime(text[:19], fmt).replace(tzinfo=timezone.utc)
        except Exception:
            pass
    return None


def is_missing(value: Any) -> bool:
    if value is None:
        return True
    if isinstance(value, float) and math.isnan(value):
        return True
    if isinstance(value, str) and value.strip() in {'', 'NA', 'N/A', 'null', 'None', 'nan'}:
        return True
    if isinstance(value, list) and len(value) == 0:
        return True
    return False


def field_has_any(field: str, tokens: Iterable[str]) -> bool:
    lower = field.lower()
    return any(token in lower for token in tokens)


def extract_timestamps(flat_record: dict[str, Any]) -> list[datetime]:
    out: list[datetime] = []
    for field, value in flat_record.items():
        if not field_has_any(field, ('time', 'date', 'timestamp', 'datetime', 'lastupdated')):
            continue
        if isinstance(value, list):
            for item in value[:1000]:
                parsed = parse_datetime(item)
                if parsed:
                    out.append(parsed)
        else:
            parsed = parse_datetime(value)
            if parsed:
                out.append(parsed)
    return out


def latlon_field_pairs(fields: Iterable[str]) -> list[tuple[str, str]]:
    field_list = list(fields)
    lats = [f for f in field_list if re.search(r'(^|[._@])lat(itude)?$', f.lower()) or f.lower().endswith('latitude')]
    lons = [f for f in field_list if re.search(r'(^|[._@])lon(gitude)?$', f.lower()) or f.lower().endswith('longitude') or f.lower().endswith('lng')]
    return [(lat, lon) for lat in lats for lon in lons]


def to_float(value: Any) -> float | None:
    try:
        if is_missing(value):
            return None
        return float(value)
    except Exception:
        return None


def candidate_key_fields(source_id: str) -> list[str]:
    mapping = {
        'londonair_monitoring': ['site_code', '@SiteCode', 'species_code', '@SpeciesCode', 'time', 'MeasurementDateGMT'],
        'openmeteo_air_quality': ['time', 'latitude', 'longitude'],
        'openaq_measurements': ['id', 'sensor.id', 'period.datetimeFrom', 'parameter.name'],
        'ncei_cdo_climate': ['station', 'date', 'datatype'],
        'waqi_air_quality': ['uid', 'idx', 'time.iso'],
        'openweather_current': ['dt', 'coord.lat', 'coord.lon'],
        'stats19_collisions': ['accident_index'],
        'naptan_stops': ['atcocode', 'ATCOCode'],
        'london_journeys': ['period_beginning', 'period_type', 'Period beginning'],
        'dft_road_traffic': ['id', 'count_point_id', 'year'],
        'tfl_transport_status': ['id', 'lineStatuses[].statusSeverity', 'lineStatuses[].created'],
        'tfl_arrivals': ['id', 'vehicleId', 'naptanId', 'expectedArrival'],
    }
    return mapping.get(source_id, [])

## 1. Source Inventory & Volume

Đọc `profile.json`, `checkpoint.json`, `request_log.jsonl` để biết đã tải gì, dung lượng bao nhiêu, source nào fail/running/success.

In [3]:
profile_paths = latest_profile_paths(DOWNLOADS_DIR, RUN_ID_FILTER)
inventory_rows: list[dict[str, Any]] = []

for profile_path in profile_paths:
    profile = read_json(profile_path, {}) or {}
    metadata_dir = metadata_dir_for_profile(profile_path)
    checkpoint = read_json(metadata_dir / 'checkpoint.json', {}) or {}
    request_log_path = metadata_dir / 'request_log.jsonl'
    request_count = 0
    http_statuses = Counter()
    if request_log_path.exists():
        for event in read_jsonl(request_log_path):
            request_count += 1
            http_statuses[str(event.get('status_code'))] += 1
    completed = checkpoint.get('completed_chunks', {}) if isinstance(checkpoint, dict) else {}
    failed = checkpoint.get('failed_chunks', {}) if isinstance(checkpoint, dict) else {}
    raw_dir = raw_dir_for_profile(profile_path)
    raw_files = [p for p in raw_dir.rglob('*') if p.is_file() and not p.name.endswith('.part')] if raw_dir.exists() else []
    formats = sorted({infer_format(p) for p in raw_files})
    inventory_rows.append({
        'source_id': profile.get('source_id') or source_id_from_profile_path(profile_path),
        'run_id': profile.get('run_id') or run_id_from_path(profile_path),
        'mode': profile.get('mode'),
        'status': profile.get('status'),
        'row_count_profile': profile.get('row_count'),
        'file_count_profile': profile.get('file_count'),
        'raw_file_count_actual': len(raw_files),
        'size_mb': profile.get('size_mb'),
        'first_timestamp': profile.get('first_timestamp'),
        'last_timestamp': profile.get('last_timestamp'),
        'failed_requests': profile.get('failed_requests'),
        'request_count': request_count,
        'http_status_counts': dict(http_statuses),
        'completed_chunks': len(completed),
        'failed_chunks': len(failed),
        'formats': ','.join(formats),
        'raw_dir': str(raw_dir.relative_to(PROJECT_ROOT)) if raw_dir.exists() else '',
        'profile_path': str(profile_path.relative_to(PROJECT_ROOT)),
    })

save_table('01_source_inventory_volume', inventory_rows)
write_json(OUTPUT_DIR / '01_source_inventory_summary.json', {
    'source_count': len(inventory_rows),
    'total_size_mb': round(sum(float(r.get('size_mb') or 0) for r in inventory_rows), 3),
    'total_rows_profile': sum(int(r.get('row_count_profile') or 0) for r in inventory_rows),
    'status_counts': dict(Counter(str(r.get('status')) for r in inventory_rows)),
})
show_table(sorted(inventory_rows, key=lambda r: float(r.get('size_mb') or 0), reverse=True), 30)

# Disk usage thực tế trên toàn bộ runtime/downloads, kể cả nhiều run_id cũ.
disk_usage_rows: list[dict[str, Any]] = []
for source_dir in sorted([p for p in DOWNLOADS_DIR.iterdir() if p.is_dir()]):
    files = [p for p in source_dir.rglob('*') if p.is_file() and not p.name.endswith('.part')]
    raw_files = [p for p in files if 'raw' in p.parts]
    metadata_files = [p for p in files if 'metadata' in p.parts]
    run_dirs = [p for p in source_dir.glob('run_id=*') if p.is_dir()]
    disk_usage_rows.append({
        'source_id': source_dir.name,
        'run_dir_count': len(run_dirs),
        'file_count_total': len(files),
        'raw_file_count_total': len(raw_files),
        'metadata_file_count_total': len(metadata_files),
        'disk_size_mb_total': round(sum(p.stat().st_size for p in files) / 1024 / 1024, 3),
        'raw_size_mb_total': round(sum(p.stat().st_size for p in raw_files) / 1024 / 1024, 3),
        'source_dir': str(source_dir.relative_to(PROJECT_ROOT)),
    })

save_table('01_downloads_disk_usage_by_source', disk_usage_rows)
write_json(OUTPUT_DIR / '01_downloads_disk_usage_summary.json', {
    'source_count_on_disk': len(disk_usage_rows),
    'total_disk_size_mb': round(sum(float(r.get('disk_size_mb_total') or 0) for r in disk_usage_rows), 3),
    'total_raw_size_mb': round(sum(float(r.get('raw_size_mb_total') or 0) for r in disk_usage_rows), 3),
    'total_file_count': sum(int(r.get('file_count_total') or 0) for r in disk_usage_rows),
    'total_raw_file_count': sum(int(r.get('raw_file_count_total') or 0) for r in disk_usage_rows),
})
show_table(sorted(disk_usage_rows, key=lambda r: float(r.get('disk_size_mb_total') or 0), reverse=True), 30)

,source_id,run_id,mode,status,row_count_profile,file_count_profile,raw_file_count_actual,size_mb,first_timestamp,last_timestamp,failed_requests,request_count,http_status_counts,completed_chunks,failed_chunks,formats,raw_dir,profile_path
0,londonair_monitoring,consolidated,consolidated,partial,None,10483,10483,435.315,1987-01-01 00:00:00,2026-05-19 23:00:00,165,10658,"{'200': 10489, 'None': 2, '400': 167}",10483,0,json,runtime\downloads\londonair_monitoring\run_id=...,runtime\downloads\londonair_monitoring\run_id=...
1,stats19_collisions,consolidated,consolidated,success,None,3,3,239.771,NaN,NaN,0,3,{'200': 3},3,0,csv,runtime\downloads\stats19_collisions\run_id=co...,runtime\downloads\stats19_collisions\run_id=co...
2,openmeteo_air_quality,consolidated,consolidated,success,None,132,132,74.136,2024-01-01T00:00,2026-05-20T23:00,0,132,{'200': 132},132,0,json,runtime\downloads\openmeteo_air_quality\run_id...,runtime\downloads\openmeteo_air_quality\run_id...
3,dft_road_traffic,consolidated,consolidated,success,None,33,33,9.526,NaN,NaN,0,36,{'200': 36},33,0,jsonl,runtime\downloads\dft_road_traffic\run_id=cons...,runtime\downloads\dft_road_traffic\run_id=cons...
4,openaq_measurements,consolidated,consolidated,partial,None,18,18,8.093,2016-01-30,2016-11-17,11,46,"{'None': 6, '200': 35, '408': 5}",18,0,jsonl,runtime\downloads\openaq_measurements\run_id=c...,runtime\downloads\openaq_measurements\run_id=c...
5,naptan_stops,consolidated,consolidated,success,None,1,1,4.287,NaN,NaN,0,2,{'200': 2},1,0,csv,runtime\downloads\naptan_stops\run_id=consolid...,runtime\downloads\naptan_stops\run_id=consolid...
6,tfl_transport_status,consolidated,consolidated,success,None,3,3,1.647,2024-07-08T07:00:00Z,2027-01-24T02:00:00Z,0,3,{'200': 3},3,0,json,runtime\downloads\tfl_transport_status\run_id=...,runtime\downloads\tfl_transport_status\run_id=...
7,openweather_current,consolidated,consolidated,success,None,99,99,0.519,NaN,NaN,0,99,{'200': 99},99,0,json,runtime\downloads\openweather_current\run_id=c...,runtime\downloads\openweather_current\run_id=c...
8,tfl_arrivals,consolidated,consolidated,success,None,3,3,0.155,2026-05-20T07:15:22.5649766Z,2026-05-20T07:44:59Z,0,3,{'200': 3},3,0,jsonl,runtime\downloads\tfl_arrivals\run_id=consolid...,runtime\downloads\tfl_arrivals\run_id=consolid...
9,ncei_cdo_climate,consolidated,consolidated,partial,None,3,3,0.143,1948-12-01,2025-08-24,6,10,"{'200': 4, '400': 5, 'None': 1}",3,0,jsonl,runtime\downloads\ncei_cdo_climate\run_id=cons...,runtime\downloads\ncei_cdo_climate\run_id=cons...


,source_id,run_dir_count,file_count_total,raw_file_count_total,metadata_file_count_total,disk_size_mb_total,raw_size_mb_total,source_dir
0,londonair_monitoring,3,20984,20968,16,885.546,870.770,runtime\downloads\londonair_monitoring
1,stats19_collisions,2,18,6,12,479.552,479.542,runtime\downloads\stats19_collisions
2,openmeteo_air_quality,3,280,264,16,148.514,148.272,runtime\downloads\openmeteo_air_quality
3,dft_road_traffic,3,84,68,16,19.482,19.427,runtime\downloads\dft_road_traffic
4,openaq_measurements,3,51,36,15,16.246,16.185,runtime\downloads\openaq_measurements
5,naptan_stops,3,19,3,16,12.871,12.860,runtime\downloads\naptan_stops
6,tfl_transport_status,2,18,6,12,3.302,3.293,runtime\downloads\tfl_transport_status
7,openweather_current,2,209,198,11,1.168,1.038,runtime\downloads\openweather_current
8,tfl_arrivals,2,18,6,12,0.319,0.310,runtime\downloads\tfl_arrivals
9,ncei_cdo_climate,3,23,7,16,0.309,0.286,runtime\downloads\ncei_cdo_climate


## 2. Heterogeneity Matrix

Mục tiêu: chứng minh các source khác nhau về format, structure, temporal grain, spatial grain, ingestion mode và join level.

In [4]:
SOURCE_TRAITS = {
    'openmeteo_air_quality': {'domain': 'environment', 'structure': 'wide hourly arrays in JSON', 'granularity': 'borough centroid x hour', 'spatial_grain': 'borough centroid', 'temporal_mode': 'historical API batch', 'join_level': 'lat/lon, borough centroid'},
    'londonair_monitoring': {'domain': 'environment', 'structure': 'nested JSON by site/species/time', 'granularity': 'site x pollutant x hour/month', 'spatial_grain': 'monitoring site', 'temporal_mode': 'historical API batch or current refresh', 'join_level': 'site code, lat/lon, borough/local authority'},
    'openaq_measurements': {'domain': 'environment', 'structure': 'JSONL measurement records', 'granularity': 'sensor x parameter x time', 'spatial_grain': 'sensor/location', 'temporal_mode': 'historical API batch/recent polling', 'join_level': 'sensor/location id, lat/lon'},
    'ncei_cdo_climate': {'domain': 'environment', 'structure': 'JSONL daily records', 'granularity': 'station x datatype x date', 'spatial_grain': 'weather station', 'temporal_mode': 'daily historical batch', 'join_level': 'station id, lat/lon if station metadata available'},
    'waqi_air_quality': {'domain': 'environment', 'structure': 'station list + nested feed JSON', 'granularity': 'station snapshot', 'spatial_grain': 'station', 'temporal_mode': 'current snapshot/polling', 'join_level': 'station uid, lat/lon'},
    'openweather_current': {'domain': 'environment', 'structure': 'current/forecast JSON', 'granularity': 'borough centroid snapshot/forecast', 'spatial_grain': 'borough centroid', 'temporal_mode': 'current snapshot/polling', 'join_level': 'lat/lon, borough centroid'},
    'stats19_collisions': {'domain': 'transport', 'structure': 'CSV files: collisions/vehicles/casualties', 'granularity': 'event/vehicle/casualty', 'spatial_grain': 'collision point', 'temporal_mode': 'validated historical batch', 'join_level': 'accident_index, lat/lon'},
    'naptan_stops': {'domain': 'transport', 'structure': 'CSV static stop reference', 'granularity': 'stop/access node', 'spatial_grain': 'stop point', 'temporal_mode': 'static/reference snapshot', 'join_level': 'ATCO code, stop coordinates'},
    'london_journeys': {'domain': 'transport', 'structure': 'CSV aggregate by reporting period/mode', 'granularity': 'period x mode', 'spatial_grain': 'London aggregate', 'temporal_mode': 'periodic aggregate file', 'join_level': 'time period, mode'},
    'dft_road_traffic': {'domain': 'transport', 'structure': 'JSONL API records', 'granularity': 'count point/year or count record', 'spatial_grain': 'road count point', 'temporal_mode': 'annual/static-ish API', 'join_level': 'count point id, road metadata, lat/lon'},
    'tfl_transport_status': {'domain': 'transport', 'structure': 'nested JSON line status/routes/disruptions', 'granularity': 'line/status snapshot', 'spatial_grain': 'line/route', 'temporal_mode': 'current snapshot/polling', 'join_level': 'line id, route'},
    'tfl_arrivals': {'domain': 'transport', 'structure': 'JSONL prediction records', 'granularity': 'stop x vehicle prediction', 'spatial_grain': 'stop point', 'temporal_mode': 'near real-time polling', 'join_level': 'Naptan id, stop id, line id'},
}

heterogeneity_rows: list[dict[str, Any]] = []
for row in inventory_rows:
    source_id = row['source_id']
    traits = SOURCE_TRAITS.get(source_id, {})
    heterogeneity_rows.append({
        'source_id': source_id,
        'domain': traits.get('domain', 'unknown'),
        'formats': row.get('formats'),
        'raw_structure': traits.get('structure', 'unknown'),
        'granularity': traits.get('granularity', 'unknown'),
        'spatial_grain': traits.get('spatial_grain', 'unknown'),
        'temporal_mode': traits.get('temporal_mode', 'unknown'),
        'join_level': traits.get('join_level', 'unknown'),
        'why_heterogeneous': f"{row.get('formats')} | {traits.get('structure', 'unknown')} | {traits.get('granularity', 'unknown')}",
    })

save_table('02_heterogeneity_matrix', heterogeneity_rows)
show_table(heterogeneity_rows, 30)

,source_id,domain,formats,raw_structure,granularity,spatial_grain,temporal_mode,join_level,why_heterogeneous
0,dft_road_traffic,transport,jsonl,JSONL API records,count point/year or count record,road count point,annual/static-ish API,"count point id, road metadata, lat/lon",jsonl | JSONL API records | count point/year o...
1,london_journeys,transport,csv,CSV aggregate by reporting period/mode,period x mode,London aggregate,periodic aggregate file,"time period, mode",csv | CSV aggregate by reporting period/mode |...
2,londonair_monitoring,environment,json,nested JSON by site/species/time,site x pollutant x hour/month,monitoring site,historical API batch or current refresh,"site code, lat/lon, borough/local authority",json | nested JSON by site/species/time | site...
3,naptan_stops,transport,csv,CSV static stop reference,stop/access node,stop point,static/reference snapshot,"ATCO code, stop coordinates",csv | CSV static stop reference | stop/access ...
4,ncei_cdo_climate,environment,jsonl,JSONL daily records,station x datatype x date,weather station,daily historical batch,"station id, lat/lon if station metadata available",jsonl | JSONL daily records | station x dataty...
5,openaq_measurements,environment,jsonl,JSONL measurement records,sensor x parameter x time,sensor/location,historical API batch/recent polling,"sensor/location id, lat/lon",jsonl | JSONL measurement records | sensor x p...
6,openmeteo_air_quality,environment,json,wide hourly arrays in JSON,borough centroid x hour,borough centroid,historical API batch,"lat/lon, borough centroid",json | wide hourly arrays in JSON | borough ce...
7,openweather_current,environment,json,current/forecast JSON,borough centroid snapshot/forecast,borough centroid,current snapshot/polling,"lat/lon, borough centroid",json | current/forecast JSON | borough centroi...
8,stats19_collisions,transport,csv,CSV files: collisions/vehicles/casualties,event/vehicle/casualty,collision point,validated historical batch,"accident_index, lat/lon",csv | CSV files: collisions/vehicles/casualtie...
9,tfl_arrivals,transport,jsonl,JSONL prediction records,stop x vehicle prediction,stop point,near real-time polling,"Naptan id, stop id, line id",jsonl | JSONL prediction records | stop x vehi...


## 3. Schema & Data Type Profiling

Đọc sample raw files, flatten nested fields, infer type, phát hiện nested/list field và type conflicts.

In [5]:
sample_records_by_source: dict[str, list[dict[str, Any]]] = {}
sample_files_by_source: dict[str, list[Path]] = {}
field_stats: dict[tuple[str, str], dict[str, Any]] = {}

for profile_path in profile_paths:
    source_id = source_id_from_profile_path(profile_path)
    raw_dir = raw_dir_for_profile(profile_path)
    files = sample_raw_files(raw_dir)
    sample_files_by_source[source_id] = files
    records: list[dict[str, Any]] = []
    for file_path in files:
        for record in read_records_from_file(file_path, source_id):
            flat = flatten(record)
            flat['_sample_file'] = str(file_path.relative_to(raw_dir)) if raw_dir.exists() else file_path.name
            records.append(flat)
            for field, value in flat.items():
                if field == '_sample_file':
                    continue
                key = (source_id, field)
                stat = field_stats.setdefault(key, {
                    'source_id': source_id,
                    'field_path': field,
                    'observed_count': 0,
                    'missing_count': 0,
                    'type_counts': Counter(),
                    'examples': [],
                    'is_nested': '.' in field or '[]' in field,
                    'is_array': False,
                })
                stat['observed_count'] += 1
                if is_missing(value):
                    stat['missing_count'] += 1
                inferred = type_name(value)
                stat['type_counts'][inferred] += 1
                if inferred.startswith('array'):
                    stat['is_array'] = True
                if len(stat['examples']) < MAX_FIELD_EXAMPLES and not is_missing(value):
                    stat['examples'].append(value)
            if len(records) >= MAX_FILES_PER_SOURCE * MAX_RECORDS_PER_FILE:
                break
    sample_records_by_source[source_id] = records

field_profile_rows: list[dict[str, Any]] = []
for stat in field_stats.values():
    type_counts = dict(stat['type_counts'])
    field_profile_rows.append({
        'source_id': stat['source_id'],
        'field_path': stat['field_path'],
        'observed_count': stat['observed_count'],
        'missing_count': stat['missing_count'],
        'missing_ratio_when_observed': round(stat['missing_count'] / max(stat['observed_count'], 1), 4),
        'type_counts': type_counts,
        'dominant_type': max(type_counts, key=type_counts.get) if type_counts else None,
        'type_conflict': len([t for t, c in type_counts.items() if c > 0]) > 1,
        'is_nested': stat['is_nested'],
        'is_array': stat['is_array'],
        'examples': stat['examples'],
    })

schema_summary_rows: list[dict[str, Any]] = []
for source_id, records in sample_records_by_source.items():
    fields = [row for row in field_profile_rows if row['source_id'] == source_id]
    schema_summary_rows.append({
        'source_id': source_id,
        'sample_file_count': len(sample_files_by_source.get(source_id, [])),
        'sample_record_count': len(records),
        'field_count': len(fields),
        'nested_field_count': sum(1 for f in fields if f['is_nested']),
        'array_field_count': sum(1 for f in fields if f['is_array']),
        'type_conflict_field_count': sum(1 for f in fields if f['type_conflict']),
        'formats_sampled': ','.join(sorted({infer_format(p) for p in sample_files_by_source.get(source_id, [])})),
    })

save_table('03_field_schema_profile', field_profile_rows)
save_table('03_source_schema_summary', schema_summary_rows)
show_table(sorted(schema_summary_rows, key=lambda r: r['field_count'], reverse=True), 30)

,source_id,sample_file_count,sample_record_count,field_count,nested_field_count,array_field_count,type_conflict_field_count,formats_sampled
0,stats19_collisions,3,3000,92,0,0,11,csv
1,openaq_measurements,18,10029,71,57,4,1,jsonl
2,tfl_transport_status,3,1409,52,34,9,8,json
3,naptan_stops,1,1000,43,0,0,8,csv
4,dft_road_traffic,33,13827,32,0,0,4,jsonl
5,tfl_arrivals,3,162,27,7,0,0,jsonl
6,londonair_monitoring,60,40680,23,0,0,5,json
7,ncei_cdo_climate,3,1241,14,0,0,0,jsonl
8,openweather_current,60,1960,14,9,0,8,json
9,london_journeys,1,209,13,0,0,12,csv


## 4. Data Quality: DAMA/IBM-style Dimensions

Dimension mapping:

- **Completeness**: missing/null/empty ratio.
- **Uniqueness**: duplicate ratio theo candidate key nếu có, fallback hash toàn record.
- **Timeliness**: latest timestamp age.
- **Validity**: timestamp parse, coordinate range, numeric parse/range proxy.
- **Consistency**: schema/type conflicts, unit conflicts, timestamp timezone patterns.
- **Accuracy**: proxy bằng impossible values: lat/lon ngoài range, pollutant/aqi âm, numeric anomaly cơ bản.

In [6]:
quality_rows: list[dict[str, Any]] = []
missing_field_rows: list[dict[str, Any]] = []

for source_id, records in sample_records_by_source.items():
    if not records:
        quality_rows.append({'source_id': source_id, 'sample_record_count': 0, 'readiness_flag': 'no_sample_records'})
        continue
    fields = sorted({field for record in records for field in record.keys() if not field.startswith('_')})
    total_cells = len(records) * max(len(fields), 1)
    missing_cells = 0
    for field in fields:
        missing = sum(1 for record in records if is_missing(record.get(field)))
        missing_cells += missing
        missing_field_rows.append({
            'source_id': source_id,
            'field_path': field,
            'sample_record_count': len(records),
            'missing_count': missing,
            'missing_ratio': round(missing / max(len(records), 1), 4),
        })
    completeness_score = 1 - (missing_cells / max(total_cells, 1))

    key_fields = [field for field in candidate_key_fields(source_id) if field in fields]
    if key_fields:
        keys = [tuple(record.get(field) for field in key_fields) for record in records]
    else:
        keys = [json.dumps(record, sort_keys=True, default=str) for record in records]
    duplicate_count = len(keys) - len(set(keys))
    uniqueness_score = 1 - duplicate_count / max(len(keys), 1)

    timestamps: list[datetime] = []
    timestamp_parse_attempts = 0
    timestamp_parse_success = 0
    for record in records:
        for field, value in record.items():
            if field_has_any(field, ('time', 'date', 'timestamp', 'datetime', 'lastupdated')):
                timestamp_parse_attempts += 1
                parsed = parse_datetime(value)
                if parsed:
                    timestamp_parse_success += 1
                    timestamps.append(parsed)
    latest_ts = max(timestamps) if timestamps else None
    freshness_age_days = None
    if latest_ts:
        freshness_age_days = round((datetime.now(timezone.utc) - latest_ts.astimezone(timezone.utc)).total_seconds() / 86400, 2)
    timeliness_score = 1.0 if freshness_age_days is not None and freshness_age_days <= 7 else 0.75 if freshness_age_days is not None and freshness_age_days <= 365 else 0.5 if freshness_age_days is not None else None

    latlon_pairs = latlon_field_pairs(fields)
    coord_attempts = 0
    coord_valid = 0
    for lat_field, lon_field in latlon_pairs[:3]:
        for record in records:
            lat = to_float(record.get(lat_field))
            lon = to_float(record.get(lon_field))
            if lat is None or lon is None:
                continue
            coord_attempts += 1
            if -90 <= lat <= 90 and -180 <= lon <= 180:
                coord_valid += 1
    timestamp_validity = timestamp_parse_success / timestamp_parse_attempts if timestamp_parse_attempts else None
    coord_validity = coord_valid / coord_attempts if coord_attempts else None
    validity_parts = [x for x in (timestamp_validity, coord_validity) if x is not None]
    validity_score = sum(validity_parts) / len(validity_parts) if validity_parts else None

    negative_pollutant_count = 0
    pollutant_attempts = 0
    for record in records:
        for field, value in record.items():
            if field_has_any(field, ('pm10', 'pm2_5', 'pm25', 'no2', 'nitrogen_dioxide', 'ozone', 'aqi')):
                number = to_float(value)
                if number is None:
                    continue
                pollutant_attempts += 1
                if number < 0:
                    negative_pollutant_count += 1
    accuracy_proxy_score = 1 - negative_pollutant_count / max(pollutant_attempts, 1) if pollutant_attempts else None

    source_fields = [row for row in field_profile_rows if row['source_id'] == source_id]
    type_conflict_count = sum(1 for row in source_fields if row['type_conflict'])
    unit_fields = [row for row in source_fields if field_has_any(row['field_path'], ('unit', 'units'))]
    consistency_score = 1 - min(type_conflict_count / max(len(source_fields), 1), 1)

    scores = [completeness_score, uniqueness_score, consistency_score]
    scores.extend([x for x in (timeliness_score, validity_score, accuracy_proxy_score) if x is not None])
    readiness_score = sum(scores) / max(len(scores), 1)

    quality_rows.append({
        'source_id': source_id,
        'sample_record_count': len(records),
        'field_count': len(fields),
        'candidate_key_fields_used': key_fields,
        'completeness_score': round(completeness_score, 4),
        'missing_cell_ratio': round(1 - completeness_score, 4),
        'uniqueness_score': round(uniqueness_score, 4),
        'duplicate_count': duplicate_count,
        'timeliness_score': None if timeliness_score is None else round(timeliness_score, 4),
        'latest_timestamp_sample': latest_ts.isoformat() if latest_ts else None,
        'freshness_age_days': freshness_age_days,
        'validity_score': None if validity_score is None else round(validity_score, 4),
        'timestamp_parse_success_ratio': None if timestamp_validity is None else round(timestamp_validity, 4),
        'coordinate_valid_ratio': None if coord_validity is None else round(coord_validity, 4),
        'consistency_score': round(consistency_score, 4),
        'type_conflict_count': type_conflict_count,
        'unit_field_count': len(unit_fields),
        'accuracy_proxy_score': None if accuracy_proxy_score is None else round(accuracy_proxy_score, 4),
        'negative_pollutant_count': negative_pollutant_count,
        'readiness_score': round(readiness_score, 4),
        'readiness_flag': 'ready_for_bronze' if readiness_score >= 0.8 else 'needs_adapter_or_review',
    })

save_table('04_quality_dimension_summary', quality_rows)
save_table('06_missing_field_profile', missing_field_rows)
show_table(sorted(quality_rows, key=lambda r: r.get('readiness_score') or 0, reverse=True), 30)

,source_id,sample_record_count,field_count,candidate_key_fields_used,completeness_score,missing_cell_ratio,uniqueness_score,duplicate_count,timeliness_score,latest_timestamp_sample,...,validity_score,timestamp_parse_success_ratio,coordinate_valid_ratio,consistency_score,type_conflict_count,unit_field_count,accuracy_proxy_score,negative_pollutant_count,readiness_score,readiness_flag
0,tfl_arrivals,162,27,"[id, vehicleId, naptanId, expectedArrival]",0.9586,0.0414,1.0000,0,1.00,2026-05-20T07:44:59+00:00,...,0.6667,0.6667,NaN,1.0000,0,0,NaN,0,0.9251,ready_for_bronze
1,dft_road_traffic,13827,32,"[id, count_point_id, year]",0.8211,0.1789,1.0000,0,NaN,NaN,...,1.0000,NaN,1.0,0.8750,4,0,NaN,0,0.9240,ready_for_bronze
2,openmeteo_air_quality,60000,11,"[time, latitude, longitude]",1.0000,0.0000,0.6000,24000,0.50,2025-02-11T15:00:00+00:00,...,1.0000,1.0000,1.0,1.0000,0,0,1.0,0,0.8500,ready_for_bronze
3,tfl_transport_status,1409,52,"[id, lineStatuses[].statusSeverity, lineStatus...",0.3950,0.6050,0.9886,16,1.00,2027-01-24T02:00:00+00:00,...,1.0000,1.0000,NaN,0.8462,8,0,NaN,0,0.8460,ready_for_bronze
4,naptan_stops,1000,43,[ATCOCode],0.5389,0.4611,1.0000,0,1.00,2026-05-18T10:37:03+00:00,...,0.8333,0.6667,1.0,0.8140,8,0,NaN,0,0.8372,ready_for_bronze
5,ncei_cdo_climate,1241,14,"[station, date, datatype]",0.3574,0.6426,1.0000,0,0.75,2025-08-24T00:00:00+00:00,...,1.0000,1.0000,1.0,1.0000,0,1,NaN,0,0.8215,ready_for_bronze
6,waqi_air_quality,348,10,[uid],0.4253,0.5747,0.1293,303,1.00,2026-05-20T15:00:00+09:00,...,1.0000,1.0000,1.0,0.9000,1,0,1.0,0,0.7424,needs_adapter_or_review
7,stats19_collisions,3000,92,[],0.3587,0.6413,1.0000,0,NaN,NaN,...,0.5000,0.0000,1.0,0.8804,11,0,NaN,0,0.6848,needs_adapter_or_review
8,openaq_measurements,10029,70,"[id, parameter.name]",0.4143,0.5857,0.0013,10016,1.00,2026-05-19T16:00:00+00:00,...,0.9999,0.9998,1.0,0.9859,1,2,NaN,0,0.6803,needs_adapter_or_review
9,london_journeys,209,13,[Period beginning],0.8811,0.1189,1.0000,0,NaN,NaN,...,NaN,NaN,NaN,0.0769,12,0,NaN,0,0.6527,needs_adapter_or_review


## 5. Temporal Coverage, Freshness & Latency

Dùng timestamp từ sample raw records và profile metadata để đánh giá coverage/freshness. Đây là estimate nhanh, không thay thế source-specific full temporal audit.

In [7]:
temporal_rows: list[dict[str, Any]] = []
for source_id, records in sample_records_by_source.items():
    timestamps: list[datetime] = []
    for record in records:
        timestamps.extend(extract_timestamps(record))
    timestamps = sorted(set(t.astimezone(timezone.utc) for t in timestamps))
    min_ts = timestamps[0] if timestamps else None
    max_ts = timestamps[-1] if timestamps else None
    coverage_days = None
    freshness_age_days = None
    inferred_grain = 'unknown'
    large_gap_count = None
    max_gap_hours = None
    if min_ts and max_ts:
        coverage_days = round((max_ts - min_ts).total_seconds() / 86400, 2)
        freshness_age_days = round((datetime.now(timezone.utc) - max_ts).total_seconds() / 86400, 2)
        diffs = [(b - a).total_seconds() / 3600 for a, b in zip(timestamps, timestamps[1:])]
        if diffs:
            median_gap = sorted(diffs)[len(diffs) // 2]
            inferred_grain = 'hourly' if median_gap <= 1.5 else 'daily' if median_gap <= 30 else 'monthly_or_irregular'
            threshold = 1.5 if inferred_grain == 'hourly' else 36 if inferred_grain == 'daily' else 900
            large_gap_count = sum(1 for gap in diffs if gap > threshold)
            max_gap_hours = round(max(diffs), 2)
    temporal_rows.append({
        'source_id': source_id,
        'timestamp_count_sample': len(timestamps),
        'min_timestamp_sample': min_ts.isoformat() if min_ts else None,
        'max_timestamp_sample': max_ts.isoformat() if max_ts else None,
        'coverage_days_sample': coverage_days,
        'freshness_age_days_sample': freshness_age_days,
        'inferred_grain_sample': inferred_grain,
        'large_gap_count_sample': large_gap_count,
        'max_gap_hours_sample': max_gap_hours,
    })

save_table('05_temporal_coverage_freshness_latency', temporal_rows)
show_table(sorted(temporal_rows, key=lambda r: r.get('coverage_days_sample') or 0, reverse=True), 30)

,source_id,timestamp_count_sample,min_timestamp_sample,max_timestamp_sample,coverage_days_sample,freshness_age_days_sample,inferred_grain_sample,large_gap_count_sample,max_gap_hours_sample
0,ncei_cdo_climate,358,1948-12-01T00:00:00+00:00,2025-08-24T00:00:00+00:00,28025.00,270.20,daily,6.0,658176.00
1,naptan_stops,922,1970-01-01T00:00:00+00:00,2026-05-18T10:37:03+00:00,20591.44,2.76,hourly,432.0,323943.84
2,londonair_monitoring,20550,1987-01-01T00:00:00+00:00,2026-05-19T23:00:00+00:00,14383.96,1.24,hourly,367.0,30696.00
3,openaq_measurements,10082,2016-01-30T00:00:00+00:00,2026-05-19T16:00:00+00:00,3762.67,1.54,hourly,26.0,59073.00
4,tfl_transport_status,123,2024-07-08T07:00:00+00:00,2027-01-24T02:00:00+00:00,929.79,-247.88,daily,39.0,5882.00
5,openmeteo_air_quality,2000,2024-01-01T00:00:00+00:00,2025-02-11T15:00:00+00:00,407.62,463.58,hourly,1.0,7785.00
6,waqi_air_quality,9,2026-05-17T01:00:00+00:00,2026-05-20T06:00:00+00:00,3.21,0.95,daily,0.0,33.00
7,tfl_arrivals,80,2026-05-20T07:15:22.564976+00:00,2026-05-20T07:44:59+00:00,0.02,0.88,hourly,0.0,0.03
8,dft_road_traffic,0,NaN,NaN,NaN,NaN,unknown,NaN,NaN
9,london_journeys,0,NaN,NaN,NaN,NaN,unknown,NaN,NaN


## 6. Missing Data & File Gap Analysis

Ngoài missing values, kiểm tra missing chunks ở mức file/path pattern: ví dụ LondonAir `site x species x month`, OpenMeteo `kind x borough`.

In [8]:
file_gap_rows: list[dict[str, Any]] = []

for profile_path in profile_paths:
    source_id = source_id_from_profile_path(profile_path)
    raw_dir = raw_dir_for_profile(profile_path)
    files = [p for p in raw_dir.rglob('*') if p.is_file() and not p.name.endswith('.part')] if raw_dir.exists() else []
    if source_id == 'londonair_monitoring':
        groups: dict[tuple[str, str, str], set[str]] = defaultdict(set)
        for path in files:
            text = str(path.relative_to(raw_dir)).replace('\\', '/')
            match = re.search(r'site=([^/]+)/species=([^/]+)/year=(\d{4})/month=(\d{2})', text)
            if match:
                site, species, year, month = match.groups()
                groups[(site, species, year)].add(month)
        for (site, species, year), months in groups.items():
            expected = {f'{m:02d}' for m in range(1, 13)}
            missing = sorted(expected - months)
            file_gap_rows.append({
                'source_id': source_id,
                'gap_scope': f'site={site}/species={species}/year={year}',
                'expected_chunks': 12,
                'observed_chunks': len(months),
                'missing_chunks': missing,
                'missing_chunk_count': len(missing),
            })
    elif source_id == 'openmeteo_air_quality':
        groups: dict[tuple[str, str, str], int] = defaultdict(int)
        for path in files:
            text = str(path.relative_to(raw_dir)).replace('\\', '/')
            match = re.search(r'kind=([^/]+)/borough=([^/]+)/year=(\d{4})/', text)
            if match:
                groups[match.groups()] += 1
        for (kind, borough, year), observed in groups.items():
            file_gap_rows.append({
                'source_id': source_id,
                'gap_scope': f'kind={kind}/borough={borough}/year={year}',
                'expected_chunks': 1,
                'observed_chunks': observed,
                'missing_chunks': [] if observed >= 1 else ['whole_year_file'],
                'missing_chunk_count': 0 if observed >= 1 else 1,
            })
    else:
        file_gap_rows.append({
            'source_id': source_id,
            'gap_scope': 'raw_file_presence',
            'expected_chunks': None,
            'observed_chunks': len(files),
            'missing_chunks': [],
            'missing_chunk_count': None,
        })

save_table('06_file_gap_analysis', file_gap_rows)
show_table(sorted(file_gap_rows, key=lambda r: r.get('missing_chunk_count') or 0, reverse=True), 30)

,source_id,gap_scope,expected_chunks,observed_chunks,missing_chunks,missing_chunk_count
0,londonair_monitoring,site=KC4/species=PM10/year=2024,12.0,2,"[03, 04, 05, 06, 07, 08, 09, 10, 11, 12]",10.0
1,londonair_monitoring,site=BX2/species=NO2/year=2026,12.0,4,"[03, 06, 07, 08, 09, 10, 11, 12]",8.0
2,londonair_monitoring,site=TD0/species=PM25/year=2026,12.0,5,"[06, 07, 08, 09, 10, 11, 12]",7.0
3,londonair_monitoring,site=TD0/species=PM10/year=2026,12.0,5,"[06, 07, 08, 09, 10, 11, 12]",7.0
4,londonair_monitoring,site=TD0/species=O3/year=2026,12.0,5,"[06, 07, 08, 09, 10, 11, 12]",7.0
5,londonair_monitoring,site=TD0/species=NO2/year=2026,12.0,5,"[06, 07, 08, 09, 10, 11, 12]",7.0
6,londonair_monitoring,site=KX4/species=PM25/year=2026,12.0,5,"[06, 07, 08, 09, 10, 11, 12]",7.0
7,londonair_monitoring,site=KX4/species=PM10/year=2026,12.0,5,"[06, 07, 08, 09, 10, 11, 12]",7.0
8,londonair_monitoring,site=KX4/species=O3/year=2026,12.0,5,"[06, 07, 08, 09, 10, 11, 12]",7.0
9,londonair_monitoring,site=KX4/species=NO2/year=2026,12.0,5,"[06, 07, 08, 09, 10, 11, 12]",7.0


## 7. Spatial Coverage & Joinability

Đánh giá source có tọa độ không, có key địa lý nào, join được ở level nào: point, station/site, stop, road segment, borough centroid, aggregate.

In [9]:
spatial_rows: list[dict[str, Any]] = []
for source_id, records in sample_records_by_source.items():
    fields = sorted({field for record in records for field in record.keys() if not field.startswith('_')})
    pairs = latlon_field_pairs(fields)
    coord_valid = 0
    coord_attempts = 0
    for lat_field, lon_field in pairs[:3]:
        for record in records:
            lat = to_float(record.get(lat_field))
            lon = to_float(record.get(lon_field))
            if lat is None or lon is None:
                continue
            coord_attempts += 1
            if -90 <= lat <= 90 and -180 <= lon <= 180:
                coord_valid += 1
    traits = SOURCE_TRAITS.get(source_id, {})
    id_like_fields = [f for f in fields if field_has_any(f, ('site', 'station', 'sensor', 'stop', 'naptan', 'atco', 'road', 'count_point', 'line', 'route', 'borough', 'localauthority'))]
    spatial_rows.append({
        'source_id': source_id,
        'spatial_grain_expected': traits.get('spatial_grain', 'unknown'),
        'join_level_expected': traits.get('join_level', 'unknown'),
        'latlon_field_pairs_detected': pairs[:10],
        'coordinate_attempts_sample': coord_attempts,
        'coordinate_valid_ratio_sample': round(coord_valid / coord_attempts, 4) if coord_attempts else None,
        'id_like_fields_detected': id_like_fields[:30],
        'joinability_flag': 'point_joinable' if coord_attempts else 'id_or_metadata_join_required' if id_like_fields else 'weak_spatial_joinability',
    })

save_table('07_spatial_coverage_joinability', spatial_rows)
show_table(spatial_rows, 30)

,source_id,spatial_grain_expected,join_level_expected,latlon_field_pairs_detected,coordinate_attempts_sample,coordinate_valid_ratio_sample,id_like_fields_detected,joinability_flag
0,dft_road_traffic,road count point,"count point id, road metadata, lat/lon","[(latitude, longitude)]",13827,1.0,"[count_point_id, end_junction_road_name, road_...",point_joinable
1,london_journeys,London aggregate,"time period, mode",[],0,NaN,[],weak_spatial_joinability
2,londonair_monitoring,monitoring site,"site code, lat/lon, borough/local authority","[(@Latitude, @Longitude)]",256,1.0,"[@LocalAuthorityCode, @LocalAuthorityName, @Si...",point_joinable
3,naptan_stops,stop point,"ATCO code, stop coordinates","[(Latitude, Longitude)]",1000,1.0,"[ATCOCode, BusStopType, NaptanCode, StopType]",point_joinable
4,ncei_cdo_climate,weather station,"station id, lat/lon if station metadata available","[(latitude, longitude)]",1,1.0,[station],point_joinable
5,openaq_measurements,sensor/location,"sensor/location id, lat/lon","[(coordinates.latitude, coordinates.longitude)]",10,1.0,"[sensors, sensors[].id, sensors[].name, sensor...",point_joinable
6,openmeteo_air_quality,borough centroid,"lat/lon, borough centroid","[(latitude, longitude)]",60000,1.0,[],point_joinable
7,openweather_current,borough centroid,"lat/lon, borough centroid",[],0,NaN,[],weak_spatial_joinability
8,stats19_collisions,collision point,"accident_index, lat/lon","[(latitude, longitude)]",999,1.0,"[first_road_class, first_road_number, pedestri...",point_joinable
9,tfl_arrivals,stop point,"Naptan id, stop id, line id",[],0,NaN,"[destinationNaptanId, lineId, lineName, naptan...",id_or_metadata_join_required


## Unit, Semantic Conflicts & Silver Planning

Phần này tạo bảng gợi ý normalize sang Silver: timestamp chính, geometry/key chính, unit conflicts, semantic naming conflicts.

In [10]:
NORMALIZATION_HINTS = {
    'openmeteo_air_quality': {'silver_table': 'silver.environment_openmeteo_hourly', 'primary_time': 'time', 'primary_geometry': 'borough centroid lat/lon', 'unit_action': 'document OpenMeteo variable units; normalize pollutant names'},
    'londonair_monitoring': {'silver_table': 'silver.environment_londonair_hourly', 'primary_time': 'measurement timestamp', 'primary_geometry': 'site coordinates/site_code', 'unit_action': 'standardize SpeciesCode to canonical pollutant names'},
    'openaq_measurements': {'silver_table': 'silver.environment_openaq_measurements', 'primary_time': 'period.datetimeFrom', 'primary_geometry': 'location/sensor coordinates', 'unit_action': 'normalize parameter/unit naming'},
    'ncei_cdo_climate': {'silver_table': 'silver.environment_ncei_daily', 'primary_time': 'date', 'primary_geometry': 'station id/coordinates', 'unit_action': 'standardize datatypes and metric units'},
    'waqi_air_quality': {'silver_table': 'silver.environment_waqi_snapshot', 'primary_time': 'time.iso', 'primary_geometry': 'station geo', 'unit_action': 'AQI is index; do not mix with concentration units'},
    'openweather_current': {'silver_table': 'silver.environment_openweather_current', 'primary_time': 'dt / poll time', 'primary_geometry': 'borough centroid lat/lon', 'unit_action': 'metric weather units; air pollution components need concentration mapping'},
    'stats19_collisions': {'silver_table': 'silver.transport_stats19_events', 'primary_time': 'date + time', 'primary_geometry': 'collision lat/lon', 'unit_action': 'decode categorical code lists'},
    'naptan_stops': {'silver_table': 'silver.transport_naptan_stops', 'primary_time': 'snapshot_date', 'primary_geometry': 'stop lat/lon', 'unit_action': 'static reference, no unit normalization'},
    'london_journeys': {'silver_table': 'silver.transport_london_journeys', 'primary_time': 'period_beginning', 'primary_geometry': 'London aggregate', 'unit_action': 'journey counts by mode'},
    'dft_road_traffic': {'silver_table': 'silver.transport_dft_road_traffic', 'primary_time': 'year', 'primary_geometry': 'count point coordinates/road id', 'unit_action': 'vehicle count metrics and AADF fields'},
    'tfl_transport_status': {'silver_table': 'silver.transport_tfl_status', 'primary_time': 'poll time / status timestamps', 'primary_geometry': 'line/route ids', 'unit_action': 'status severity code mapping'},
    'tfl_arrivals': {'silver_table': 'silver.transport_tfl_arrivals', 'primary_time': 'expectedArrival / poll time', 'primary_geometry': 'Naptan stop id', 'unit_action': 'prediction timing in UTC'},
}

semantic_rows: list[dict[str, Any]] = []
for source_id in sample_records_by_source:
    fields = [row['field_path'] for row in field_profile_rows if row['source_id'] == source_id]
    pollutant_fields = [f for f in fields if field_has_any(f, ('pm10', 'pm2_5', 'pm25', 'no2', 'nitrogen_dioxide', 'ozone', 'o3', 'aqi'))]
    unit_fields = [f for f in fields if field_has_any(f, ('unit', 'units'))]
    timestamp_fields = [f for f in fields if field_has_any(f, ('time', 'date', 'timestamp', 'datetime'))]
    hints = NORMALIZATION_HINTS.get(source_id, {})
    semantic_rows.append({
        'source_id': source_id,
        'silver_table_hint': hints.get('silver_table'),
        'primary_time_hint': hints.get('primary_time'),
        'primary_geometry_hint': hints.get('primary_geometry'),
        'unit_action_hint': hints.get('unit_action'),
        'pollutant_or_index_fields_detected': pollutant_fields[:40],
        'unit_fields_detected': unit_fields[:20],
        'timestamp_fields_detected': timestamp_fields[:30],
        'semantic_conflict_notes': 'pollutant/index naming needs canonical dictionary' if pollutant_fields else '',
    })

save_table('08_unit_semantic_conflicts_and_silver_plan', semantic_rows)
show_table(semantic_rows, 30)

,source_id,silver_table_hint,primary_time_hint,primary_geometry_hint,unit_action_hint,pollutant_or_index_fields_detected,unit_fields_detected,timestamp_fields_detected,semantic_conflict_notes
0,dft_road_traffic,silver.transport_dft_road_traffic,year,count point coordinates/road id,vehicle count metrics and AADF fields,[],[],[],
1,london_journeys,silver.transport_london_journeys,period_beginning,London aggregate,journey counts by mode,[],[],[],
2,londonair_monitoring,silver.environment_londonair_hourly,measurement timestamp,site coordinates/site_code,standardize SpeciesCode to canonical pollutant...,[],[],"[@DateClosed, @DateOpened, @MeasurementDateGMT]",
3,naptan_stops,silver.transport_naptan_stops,snapshot_date,stop lat/lon,"static reference, no unit normalization",[],[],"[DefaultWaitTime, CreationDateTime, Modificati...",
4,ncei_cdo_climate,silver.environment_ncei_daily,date,station id/coordinates,standardize datatypes and metric units,[],[elevationUnit],"[mindate, maxdate, date]",
5,openaq_measurements,silver.environment_openaq_measurements,period.datetimeFrom,location/sensor coordinates,normalize parameter/unit naming,[],"[sensors[].parameter.units, parameter.units]","[timezone, licenses[].dateFrom, licenses[].dat...",
6,openmeteo_air_quality,silver.environment_openmeteo_hourly,time,borough centroid lat/lon,document OpenMeteo variable units; normalize p...,"[pm10, pm2_5, nitrogen_dioxide, ozone, europea...",[],[time],pollutant/index naming needs canonical dictionary
7,openweather_current,silver.environment_openweather_current,dt / poll time,borough centroid lat/lon,metric weather units; air pollution components...,"[main.aqi, components.no2, components.o3, comp...",[],[],pollutant/index naming needs canonical dictionary
8,stats19_collisions,silver.transport_stats19_events,date + time,collision lat/lon,decode categorical code lists,[],[],"[date, time]",
9,tfl_arrivals,silver.transport_tfl_arrivals,expectedArrival / poll time,Naptan stop id,prediction timing in UTC,[],[],"[timestamp, timeToStation, timeToLive]",


## Final Report & Output Manifest

Cell cuối lưu `summary_report.md` và `analysis_manifest.json` để tiện mở lại sau này.

In [11]:
output_files = sorted(p for p in OUTPUT_DIR.glob('*') if p.is_file())
summary = read_json(OUTPUT_DIR / '01_source_inventory_summary.json', {}) or {}
disk_summary = read_json(OUTPUT_DIR / '01_downloads_disk_usage_summary.json', {}) or {}
status_counts = summary.get('status_counts', {})
top_sources = sorted(inventory_rows, key=lambda r: float(r.get('size_mb') or 0), reverse=True)[:8]
needs_review = [row for row in quality_rows if row.get('readiness_flag') != 'ready_for_bronze']

report_lines = [
    '# NEXUS Downloaded Data Profiling Report',
    '',
    f'- Analysis ID: `{ANALYSIS_ID}`',
    f'- Downloads dir: `{DOWNLOADS_DIR}`',
    f'- Output dir: `{OUTPUT_DIR}`',
    f"- Download disk size MB: {disk_summary.get('total_disk_size_mb')}",
    f"- Raw disk size MB: {disk_summary.get('total_raw_size_mb')}",
    f"- Download file count: {disk_summary.get('total_file_count')}",
    f"- Source count: {summary.get('source_count')}",
    f"- Total size MB: {summary.get('total_size_mb')}",
    f"- Total row count from profiles: {summary.get('total_rows_profile')}",
    f'- Status counts: `{status_counts}`',
    '',
    '## Largest Sources',
]
for row in top_sources:
    report_lines.append(f"- `{row['source_id']}`: {row.get('size_mb')} MB, rows={row.get('row_count_profile')}, status={row.get('status')}")

report_lines.extend(['', '## Sources Needing Review'])
if needs_review:
    for row in needs_review:
        report_lines.append(f"- `{row['source_id']}`: readiness={row.get('readiness_score')}, flag={row.get('readiness_flag')}")
else:
    report_lines.append('- None in sampled profiling.')

report_lines.extend([
    '',
    '## Suggested Next Steps',
    '- Use `01_downloads_disk_usage_by_source.csv` to track actual local disk usage across all runs.',
    '- Use `01_source_inventory_volume.csv` to decide selected-run storage budget and source completeness.',
    '- Use `03_field_schema_profile.csv` to design Bronze raw envelopes and Silver adapters.',
    '- Use `04_quality_dimension_summary.csv` and `06_missing_field_profile.csv` to define data contracts.',
    '- Use `07_spatial_coverage_joinability.csv` to decide geospatial enrichment level.',
    '- Use `08_unit_semantic_conflicts_and_silver_plan.csv` to plan unit and schema standardization.',
])

report_path = OUTPUT_DIR / 'summary_report.md'
report_path.write_text('\n'.join(report_lines), encoding='utf-8')

manifest = {
    'analysis_id': ANALYSIS_ID,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'project_root': str(PROJECT_ROOT),
    'downloads_dir': str(DOWNLOADS_DIR),
    'output_dir': str(OUTPUT_DIR),
    'run_id_filter': RUN_ID_FILTER,
    'parameters': {
        'max_files_per_source': MAX_FILES_PER_SOURCE,
        'max_records_per_file': MAX_RECORDS_PER_FILE,
        'max_flatten_depth': MAX_FLATTEN_DEPTH,
    },
    'outputs': [str(path.relative_to(OUTPUT_DIR)) for path in output_files] + ['summary_report.md'],
}
write_json(OUTPUT_DIR / 'analysis_manifest.json', manifest)

print(f'Saved report: {report_path}')
print(f'Saved manifest: {OUTPUT_DIR / "analysis_manifest.json"}')
print('Output files:')
for path in sorted(OUTPUT_DIR.glob('*')):
    if path.is_file():
        print(' -', path.name)

Saved report: d:\.Kỳ II năm Ba\Big Data\NEXUS\runtime\analysis\downloaded_data_profile\analysis_id=20260521T045042Z\summary_report.md
Saved manifest: d:\.Kỳ II năm Ba\Big Data\NEXUS\runtime\analysis\downloaded_data_profile\analysis_id=20260521T045042Z\analysis_manifest.json
Output files:
 - 01_downloads_disk_usage_by_source.csv
 - 01_downloads_disk_usage_by_source.json
 - 01_downloads_disk_usage_summary.json
 - 01_source_inventory_summary.json
 - 01_source_inventory_volume.csv
 - 01_source_inventory_volume.json
 - 02_heterogeneity_matrix.csv
 - 02_heterogeneity_matrix.json
 - 03_field_schema_profile.csv
 - 03_field_schema_profile.json
 - 03_source_schema_summary.csv
 - 03_source_schema_summary.json
 - 04_quality_dimension_summary.csv
 - 04_quality_dimension_summary.json
 - 05_temporal_coverage_freshness_latency.csv
 - 05_temporal_coverage_freshness_latency.json
 - 06_file_gap_analysis.csv
 - 06_file_gap_analysis.json
 - 06_missing_field_profile.csv
 - 06_missing_field_profile.json
 - 0